[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-forever/FRIDA-Decisions/blob/grok/examples-citation_check/examples/citation_check/citation_check.ipynb)

# Check a claim against the source

A citation is either supported, contradicted, or not in the source. An LLM asked "is this true?" tends to answer in prose and to fill gaps. FRIDA-Decisions gets the source as the state and the claim inside one `choice` question, and code sends low-confidence verdicts to a person.

Keep the claim short. Instructions are cut at 96 tokens, including a short suffix the model was trained with, so a long quotation in the instruction is truncated.

Pattern: [Double-checking citations](https://docs.typesafe.ai/cookbooks/citation_check).


In [1]:
import sys
!{sys.executable} -m pip install -q "frida-decisions[torch] @ git+https://github.com/ai-forever/FRIDA-Decisions@v0.4.0"


In [2]:
import json, os
from pathlib import Path
import torch
import frida_decisions
from frida_decisions import Judge

MODEL = os.environ.get("FD_MODEL", "ai-forever/FRIDA-Decisions")   # repo id or a local folder
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(min(4, os.cpu_count() or 1))
judge = Judge.from_pretrained(MODEL, device=device)
gpu = f" ({torch.cuda.get_device_name(0)})" if device == "cuda" else ""
print(f"frida-decisions {frida_decisions.__version__}, loaded on {device}{gpu}, {judge.dtype}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

frida-decisions 0.4.0, loaded on cpu, torch.float32


In [3]:
VERDICTS = {
    "supports": "источник подтверждает утверждение",
    "contradicts": "источник противоречит утверждению",
    "insufficient": "в источнике нет данных, чтобы судить",
}
# Below this, the choice is shown but a person decides. Several correct choices
# on this set sit around 0.3, so the cutoff is doing real work.
MIN_CONFIDENCE = 0.5


def request_for(source: str, claim: str) -> dict:
    return {
        "state": source,
        "questions": {"verdict": {
            "type": "choice",
            "instructions": "Утверждение: «" + claim + "». Как к нему относится источник?",
            "criteria": VERDICTS,
        }},
    }


def decide(answer: dict) -> dict:
    if answer["confidence"] < MIN_CONFIDENCE:
        return {"verdict": "review", "choice": answer["choice"], "confidence": answer["confidence"]}
    return {"verdict": answer["choice"], "choice": answer["choice"], "confidence": answer["confidence"]}


## A third option for "the source does not say"

`review` is not a model label. It is the branch taken when confidence is below the cutoff, so an uncertain `supports` is not stored as a fact.


In [4]:
def run(judge, rows: list[dict]) -> None:
    choice_hits = auto = auto_hits = 0
    for row in rows:
        response = judge(request_for(row["source"], row["claim"]))
        answer = response["answers"]["verdict"]
        decision = decide(answer)
        choice_ok = answer["choice"] == row["verdict"]
        choice_hits += choice_ok
        if decision["verdict"] == "review":
            mark = "review"
        else:
            auto += 1
            auto_hits += decision["verdict"] == row["verdict"]
            mark = "ok" if decision["verdict"] == row["verdict"] else "miss"
        print(f"{mark:6} choice {answer['choice']:14} gold {row['verdict']:14} "
              f"p={answer['confidence']:.2f}  action {decision['verdict']}")
        print(f"       {row['claim']}")
    print(f"\nchoice agrees with the label in {choice_hits}/{len(rows)}")
    if auto:
        print(f"auto at confidence >= {MIN_CONFIDENCE}: {auto}/{len(rows)}, "
              f"accuracy {auto_hits}/{auto}")
    else:
        print(f"nothing cleared confidence {MIN_CONFIDENCE}")


In [5]:
def load_rows():
    import urllib.request
    path = Path("data.json")
    if not path.exists():
        url = ("https://raw.githubusercontent.com/ai-forever/FRIDA-Decisions/"
               "grok/examples-citation-check/examples/citation_check/data.json")
        path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
    return json.loads(path.read_text(encoding="utf-8"))

run(judge, load_rows())


ok     choice supports       gold supports       p=0.86  action supports
       Качественный товар можно вернуть в течение 14 дней при наличии чека.
review choice contradicts    gold contradicts    p=0.35  action review
       Вернуть качественный товар можно в любой срок и без чека.
review choice contradicts    gold insufficient   p=0.22  action review
       Доставка по городу бесплатная.
ok     choice supports       gold supports       p=0.53  action supports
       Заявку на обмен после 18:00 рассмотрят на следующий рабочий день.
review choice contradicts    gold contradicts    p=0.27  action review
       Обмен работает круглосуточно, включая выходные.
ok     choice contradicts    gold contradicts    p=0.69  action contradicts
       Если колба треснула от удара, гарантия это покрывает.
review choice supports       gold supports       p=0.29  action review
       Гарантия на чайник длится один год.
review choice contradicts    gold insufficient   p=0.33  action review
       В ком